# ETA Package Test

End-to-end test of the `eta-clf` package using 1 000 positive + 1 000 negative protein sequences.

**Runtime**: set to **GPU (T4)** — `Runtime → Change runtime type → T4 GPU`

## Workflow
1. Run **Cell 1 (Install)** and then **restart the runtime** (`Runtime → Restart session`)
2. After restart, run all remaining cells top to bottom — **skip Cell 1**

> The restart is required because the `esm` package downgrades numpy, and Python
> needs a fresh start to load the corrected numpy binary.

## 1. Install
**Run this cell once, then restart the runtime. Do not re-run after restart.**

In [ ]:
# Install etap-clf from GitHub
!pip install -q "git+https://github.com/Sitgttish/summer26.git#subdirectory=eta_package"

# The esm package downgrades numpy to 1.26; restore a compatible 2.x version
!pip install -q --force-reinstall "numpy>=2.0"

print()
print("Installation complete.")
print("ACTION REQUIRED: Restart the runtime now — Runtime > Restart session")
print("After restart, run all cells BELOW this one (skip this install cell).")

---
## 2. Download test data
*(Start here after restarting the runtime)*

In [ ]:
import os
os.makedirs('test_data', exist_ok=True)

BASE = 'https://raw.githubusercontent.com/Sitgttish/summer26/main/eta_package/test_data'
!wget -q -O test_data/test_pos.fasta {BASE}/test_pos.fasta
!wget -q -O test_data/test_neg.fasta {BASE}/test_neg.fasta

n_pos = sum(1 for l in open('test_data/test_pos.fasta') if l.startswith('>'))
n_neg = sum(1 for l in open('test_data/test_neg.fasta') if l.startswith('>'))
print(f'Positive sequences : {n_pos}')
print(f'Negative sequences : {n_neg}')
print(f'Total              : {n_pos + n_neg}')

## 3. HuggingFace authentication

ESM3 is a gated model. You need to:
1. Accept the license at https://huggingface.co/EvolutionaryScale/esm3-sm-open-v1
2. Create a read token at https://huggingface.co/settings/tokens
3. Add it as a Colab Secret named `HF_TOKEN` (**Runtime → Manage secrets**) — or paste it below.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets.')
except Exception:
    os.environ['HF_TOKEN'] = 'hf_YOUR_TOKEN_HERE'   # <-- paste token here if not using Secrets
    print('HF_TOKEN set manually.')

## 4. Training

Trains on 1 000 positive + 1 000 negative sequences.  
ESM3 embeds all sequences first (~5–10 min on T4), then ETA trains on the cached embeddings.

**Expected output**: `best_model.pth`, `training_history.csv`, `test_metrics.csv` in `./model_output/`.

In [ ]:
!eta --train test_data/test_pos.fasta test_data/test_neg.fasta ./model_output/ \
    --epochs 15 --patience 5 --embed-batch-size 16

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.read_csv('model_output/training_history.csv')
print(history.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['epoch'], history['loss'], marker='o', color='steelblue')
axes[0].set_title('Training loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(alpha=0.3)

axes[1].plot(history['epoch'], history['val_auc'], marker='o', color='darkorange')
axes[1].set_title('Validation AUC'); axes[1].set_xlabel('Epoch')
axes[1].set_ylim(0, 1); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('\nTest set metrics:')
print(pd.read_csv('model_output/test_metrics.csv').T.rename(columns={0: 'value'}).to_string())

## 5. Inference with metrics

Create a small labeled evaluation FASTA (100 positive + 100 negative) so the CLI
can report accuracy, AUC, sensitivity and specificity.

In [ ]:
def take_labeled(src, label, dst, n=100):
    """Copy first n records from src FASTA and append |label=<label> to each header."""
    count = 0
    header, seq_lines = None, []
    with open(src) as fin, open(dst, 'w') as fout:
        for line in fin:
            if line.startswith('>'):
                if header:
                    fout.write(header)
                    fout.writelines(seq_lines)
                if count >= n:
                    break
                header = f"{line.rstrip()}|label={label}\n"
                seq_lines = []
                count += 1
            else:
                seq_lines.append(line)
        if header and count <= n:
            fout.write(header)
            fout.writelines(seq_lines)

take_labeled('test_data/test_pos.fasta', 1, 'eval_pos.fasta', n=100)
take_labeled('test_data/test_neg.fasta', 0, 'eval_neg.fasta', n=100)

with open('eval_labeled.fasta', 'w') as fout:
    for f in ['eval_pos.fasta', 'eval_neg.fasta']:
        fout.write(open(f).read())

n = sum(1 for l in open('eval_labeled.fasta') if l.startswith('>'))
print(f'Labeled eval FASTA: {n} sequences (100 positive + 100 negative)')

In [ ]:
!eta --eval model_output/best_model.pth eval_labeled.fasta ./results/predictions.csv

In [ ]:
import pandas as pd

results = pd.read_csv('results/predictions.csv')
print(f'Total predictions : {len(results)}')
print(f'Predicted positive: {results["predicted_label"].sum()}')
print(f'Predicted negative: {(results["predicted_label"] == 0).sum()}')
print(f'\nFirst 10 rows:')
results.head(10)

## 6. Attention analysis

Re-run inference with `--gene-analyze` to generate five attention-weight plots:
1. Mean attention per amino acid (by class)
2. Positive-class attention enrichment (log₂ ratio)
3. Top-20 high-attention 5-mer motifs
4. Gene-level attention heatmap
5. Positional attention profile (N→C terminus)

In [ ]:
!eta --eval model_output/best_model.pth eval_labeled.fasta ./results/predictions.csv \
    --gene-analyze --analyze-dir ./results/analysis/

In [ ]:
from IPython.display import Image, display
import glob

for img_path in sorted(glob.glob('results/analysis/*.png')):
    print(f'\n{img_path}')
    display(Image(img_path, width=900))

## 7. Python API

The same pipeline is available as a Python API for integration into custom scripts or notebooks.

In [ ]:
import os
from eta import run_eval

results_df = run_eval(
    model_path='model_output/best_model.pth',
    sequences_fasta='eval_labeled.fasta',
    output_path='./api_results.csv',
    hf_token=os.environ.get('HF_TOKEN'),
    gene_analyze=False,
)

print(f'Returned DataFrame shape: {results_df.shape}')
print(f'Mean probability (positive class): {results_df["prob_positive"].mean():.4f}')
results_df.head()

In [ ]:
# Training via Python API (commented out — already trained above)

# from eta import run_training
# ckpt_path, metrics = run_training(
#     pos_fasta='test_data/test_pos.fasta',
#     neg_fasta='test_data/test_neg.fasta',
#     output_dir='./api_model_output/',
#     hparams={'max_epochs': 10, 'patience': 5},
#     hf_token=os.environ.get('HF_TOKEN'),
# )
# print(ckpt_path)
# print(metrics)
print('Uncomment the block above to retrain via the Python API.')